# Skin Lesion **Detection** — PAD-UFES-20 (YOLOv8)
**Week 3 deliverable:** detection model + prediction outputs + mAP results.

Project 1 (Patient Skin Lesion Monitoring), Computer Vision Semester 6.

PAD-UFES-20 ships with **classification labels only** (no bounding boxes).
Each clinical image is a close-up of a single lesion, so we **auto-generate
bounding boxes** via classical image processing (LAB color + Otsu + largest
central blob) and use the dataset's `diagnostic` field as the box class.
Then we train **YOLOv8-nano** on those pseudo-labels.

## How to use
1. Runtime → Change runtime type → **GPU (T4)** → Save.
2. Make sure `images.zip` and `metadata.csv` are in your Drive folder
   `skin-lesion-monitoring-cv_dataset` (same folder used for classification).
3. Run top to bottom. Best weights are saved to `pad-ufes-20-results/best_yolov8.pt`.
4. Download that `.pt` file next to `detect_app.py` to run the local web app.

In [ ]:
# 1. Imports + GPU check
import os, glob, zipfile, time, shutil, random
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime -> Change runtime type -> GPU (T4).')

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# 3. Config
DATASET_DIR   = '/content/drive/MyDrive/skin-lesion-monitoring-cv_dataset'
ZIP_ON_DRIVE  = os.path.join(DATASET_DIR, 'images.zip')
META_ON_DRIVE = os.path.join(DATASET_DIR, 'metadata.csv')
DATA_ROOT     = '/content/pad-ufes-20'
YOLO_ROOT     = '/content/yolo_data'
OUT_DIR       = '/content/drive/MyDrive/pad-ufes-20-results'

IMG_SIZE   = 640
EPOCHS     = 25
BATCH_SIZE = 16
VAL_SPLIT  = 0.2
SEED       = 42
CLASSES    = ['BCC', 'SCC', 'ACK', 'SEK', 'MEL', 'NEV']

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.isfile(ZIP_ON_DRIVE),  f'Zip not found: {ZIP_ON_DRIVE}'
assert os.path.isfile(META_ON_DRIVE), f'metadata.csv not found: {META_ON_DRIVE}'
print('Zip OK :', ZIP_ON_DRIVE)
print('Meta OK:', META_ON_DRIVE)

In [ ]:
# 4. Copy zip + metadata to local disk, then extract
local_zip  = '/content/images.zip'
local_meta = '/content/metadata.csv'

t = time.time()
shutil.copy(ZIP_ON_DRIVE, local_zip)
shutil.copy(META_ON_DRIVE, local_meta)
print(f'Copied zip+csv in {time.time() - t:.0f}s')

t = time.time()
os.makedirs(DATA_ROOT, exist_ok=True)
with zipfile.ZipFile(local_zip) as z:
    z.extractall(DATA_ROOT)
shutil.copy(local_meta, os.path.join(DATA_ROOT, 'metadata.csv'))
print(f'Extracted in {time.time() - t:.0f}s')

paths = {}
for ext in ('*.png', '*.jpg', '*.jpeg'):
    for p in glob.glob(os.path.join(DATA_ROOT, '**', ext), recursive=True):
        paths[os.path.basename(p)] = p
print('Indexed images:', len(paths))

meta = pd.read_csv(os.path.join(DATA_ROOT, 'metadata.csv'))
meta = meta[['img_id', 'diagnostic']].dropna()
meta = meta[meta['diagnostic'].isin(CLASSES)].copy()
meta['path'] = meta['img_id'].map(paths)
meta = meta.dropna(subset=['path']).reset_index(drop=True)
print('Usable samples:', len(meta))
print(meta['diagnostic'].value_counts())

In [ ]:
# 5. Auto-generate bounding boxes (image processing)
#    PAD-UFES-20 has no boxes -> derive one box per image:
#      LAB color -> L channel -> blur -> Otsu (inverted) -> close -> largest
#      connected component (favor centered ones) -> bounding rect.

def auto_bbox(img_bgr, min_area_frac=0.005, max_area_frac=0.92):
    """Return (x1,y1,x2,y2) in pixel coords, or None."""
    h, w = img_bgr.shape[:2]
    lab  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    L = cv2.GaussianBlur(lab[..., 0], (7, 7), 0)
    # invert so darker lesion becomes high values
    Linv = 255 - L
    _, th = cv2.threshold(Linv, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, k, iterations=2)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN,  k, iterations=1)

    n, lab_im, stats, cents = cv2.connectedComponentsWithStats(th, 8)
    if n <= 1:
        return None
    img_cx, img_cy = w / 2, h / 2
    best, best_score = None, -1.0
    for i in range(1, n):
        x, y, bw, bh, area = stats[i]
        af = area / (w * h)
        if af < min_area_frac or af > max_area_frac:
            continue
        cx, cy = cents[i]
        d = np.hypot(cx - img_cx, cy - img_cy) / np.hypot(img_cx, img_cy)
        # bigger blobs + closer to center = higher score
        score = af * (1.0 - 0.6 * d)
        if score > best_score:
            best, best_score = (x, y, bw, bh), score
    if best is None:
        return None
    x, y, bw, bh = best
    # widen the box a touch to include lesion border
    pad = int(0.05 * max(bw, bh))
    x1 = max(0, x - pad); y1 = max(0, y - pad)
    x2 = min(w - 1, x + bw + pad); y2 = min(h - 1, y + bh + pad)
    return x1, y1, x2, y2

# Try on a few samples + visualize
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, row in zip(axes.ravel(), meta.sample(8, random_state=SEED).itertuples()):
    im = cv2.imread(row.path)
    b  = auto_bbox(im)
    rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    if b:
        x1, y1, x2, y2 = b
        cv2.rectangle(rgb, (x1, y1), (x2, y2), (0, 255, 0), 6)
    ax.imshow(rgb); ax.set_title(row.diagnostic); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 6. Build YOLOv8 dataset (images + labels in train/val splits)
from sklearn.model_selection import train_test_split

for split in ('train', 'val'):
    os.makedirs(f'{YOLO_ROOT}/images/{split}', exist_ok=True)
    os.makedirs(f'{YOLO_ROOT}/labels/{split}', exist_ok=True)

cls_to_id = {c: i for i, c in enumerate(CLASSES)}
train_df, val_df = train_test_split(
    meta, test_size=VAL_SPLIT, stratify=meta['diagnostic'], random_state=SEED)

def write_split(df, split):
    kept, skipped = 0, 0
    for r in df.itertuples():
        im = cv2.imread(r.path)
        if im is None:
            skipped += 1; continue
        h, w = im.shape[:2]
        b = auto_bbox(im)
        if b is None:
            skipped += 1; continue
        x1, y1, x2, y2 = b
        xc = ((x1 + x2) / 2) / w
        yc = ((y1 + y2) / 2) / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h
        cid = cls_to_id[r.diagnostic]
        # symlink image (cheap); write label
        dst_img = f'{YOLO_ROOT}/images/{split}/{r.img_id}'
        if not os.path.exists(dst_img):
            try:
                os.symlink(r.path, dst_img)
            except Exception:
                shutil.copy(r.path, dst_img)
        stem = os.path.splitext(r.img_id)[0]
        with open(f'{YOLO_ROOT}/labels/{split}/{stem}.txt', 'w') as f:
            f.write(f'{cid} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n')
        kept += 1
    return kept, skipped

tr_k, tr_s = write_split(train_df, 'train')
va_k, va_s = write_split(val_df,   'val')
print(f'train: kept {tr_k}, skipped {tr_s}')
print(f'val  : kept {va_k}, skipped {va_s}')

yaml_text = (
    f'path: {YOLO_ROOT}\n'
    f'train: images/train\n'
    f'val: images/val\n'
    f'nc: {len(CLASSES)}\n'
    f'names: {CLASSES}\n'
)
yaml_path = f'{YOLO_ROOT}/data.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_text)
print('\n--- data.yaml ---\n' + yaml_text)

In [ ]:
# 7. Install Ultralytics YOLOv8
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'ultralytics'], check=True)
from ultralytics import YOLO
import ultralytics
print('ultralytics', ultralytics.__version__)

In [ ]:
# 8. Train YOLOv8-nano (transfer from COCO weights)
model = YOLO('yolov8n.pt')   # downloads pretrained weights
results = model.train(
    data    = yaml_path,
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    device  = 0 if device == 'cuda' else 'cpu',
    project = '/content/runs_det',
    name    = 'lesion_yolov8n',
    seed    = SEED,
    patience= 8,
    plots   = True,
)
best_pt = '/content/runs_det/lesion_yolov8n/weights/best.pt'
print('Best weights:', best_pt, '  exists:', os.path.isfile(best_pt))

In [ ]:
# 9. Validate -> mAP metrics, then save best weights to Drive
best = YOLO(best_pt)
metrics = best.val(data=yaml_path, imgsz=IMG_SIZE, plots=True)

print('\n=== mAP results ===')
print(f'mAP@50    : {metrics.box.map50:.4f}')
print(f'mAP@50-95 : {metrics.box.map:.4f}')
print(f'precision : {metrics.box.mp:.4f}')
print(f'recall    : {metrics.box.mr:.4f}')

out_pt = os.path.join(OUT_DIR, 'best_yolov8.pt')
shutil.copy(best_pt, out_pt)
print('\nSaved to Drive:', out_pt)

# Save a metrics text file for the report
with open(os.path.join(OUT_DIR, 'metrics_yolov8.txt'), 'w') as f:
    f.write('YOLOv8n  PAD-UFES-20 detection (auto-bbox labels)\n')
    f.write(f'mAP@50    : {metrics.box.map50:.4f}\n')
    f.write(f'mAP@50-95 : {metrics.box.map:.4f}\n')
    f.write(f'precision : {metrics.box.mp:.4f}\n')
    f.write(f'recall    : {metrics.box.mr:.4f}\n')

In [ ]:
# 10. Visualize predictions on a few val images
val_imgs = sorted(glob.glob(f'{YOLO_ROOT}/images/val/*'))[:8]
preds = best.predict(val_imgs, imgsz=IMG_SIZE, conf=0.25, verbose=False)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, r in zip(axes.ravel(), preds):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.tight_layout(); plt.show()
print('Done. Download best_yolov8.pt from Drive into the project root for the local app.')